## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)

In [3]:
## Connection
connection = presto.connect(
        host = 'bi-presto.serving.data.production.internal' ,
        #host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Methodology
### Objective: 
- New UI scratch card direct open - learnings for visibility

### Segmentation Criteria: 
- Customer Entry Criteria (Bangalore  <> Link & Auto & Cab) 
- Customer Metrics (LTR <> FE <> RR <> NET <> Segment <> COBRA <> COBRM <> OCARA <> expired)

### Collect Data: 
- fare_estimates_enriched <> order_logs_snapshot/immutable/fact <> iallocator_customer_segments
- Create a customer-level aggregated view.

### Stratified Random Sampling: 
- Use randomization to assign customers within each segment to either the control group (A) or the treatment group (B).
- <b>Balance Groups:</b>     
    - Ensure that the size of the groups is balanced. 
    - Conduct a pre-analysis to check if the groups are indeed homogeneous before starting the experiment.
    - This helps in confirming that the segmentation strategy is effective.
- <b>Exclude Outliers:</b>
    - Identify and exclude outliers that might skew the results. 
    - Outliers could be customers with extremely high or low values in the chosen criteria that are not representative of the majority.
	

### Implement the Experiment: 
- Implement the changes for the treatment group and keep the control group unchanged. 
	
### Monitor and Analyse: 
- Monitor the experiment in real-time and, once completed, analyse the results to determine if there are significant differences between the groups.

## Parameter

In [4]:
current_working_directory = os.getcwd()
print(current_working_directory)

/Users/E2074/analytics/customer/Ads-Monetisation/scratchcard-direct-open


In [5]:
# local_datasets = '/Users/rapido/local-datasets/customer/ads-monetisation/captian_dispatch_exp/'

In [6]:
## Parameter 
start_date = '20240501'
end_date = '20240626'
cities = ['Bangalore']
services = ['Link','Auto','Bike Lite','CabPremium','CabEconomy']

In [7]:
# Convert the lists to comma-separated strings
city_str = "', '".join(cities)
service_str = "', '".join(services)
print(city_str)
print(service_str)

Bangalore
Link', 'Auto', 'Bike Lite', 'CabPremium', 'CabEconomy


In [8]:
service_mapping_query = f"""
    select 
        distinct service_detail_id, service_name
    from 
        pricing.fare_estimates_enriched
    where 
        yyyymmdd = '20240625'
        and city IN ('{city_str}')
        and service_name IN ('{service_str}')
"""

df_service_mapping = pd.read_sql(service_mapping_query, connection)
df_service_mapping

,service_detail_id,service_name
0,5c53562fceb6fc9241980547,Auto
1,664c599092b1525397396c67,CabPremium
2,64d22dc048d4848cb575fe8f,Bike Lite
3,57370b61a6855d70057417d1,Link
4,6516a74cb0ef8abf16cec40d,CabEconomy


In [9]:
service_detail_id_list = df_service_mapping.service_detail_id.tolist()
service_detail_id_str = "', '".join(service_detail_id_list)
print(service_detail_id_list)
print(service_detail_id_str)

['5c53562fceb6fc9241980547', '664c599092b1525397396c67', '64d22dc048d4848cb575fe8f', '57370b61a6855d70057417d1', '6516a74cb0ef8abf16cec40d']
5c53562fceb6fc9241980547', '664c599092b1525397396c67', '64d22dc048d4848cb575fe8f', '57370b61a6855d70057417d1', '6516a74cb0ef8abf16cec40d


## Datasets

### Fetching data

In [10]:
service_detail_id_list

['5c53562fceb6fc9241980547',
 '664c599092b1525397396c67',
 '64d22dc048d4848cb575fe8f',
 '57370b61a6855d70057417d1',
 '6516a74cb0ef8abf16cec40d']

##customer_funnel

customer_funnel = f"""

    WITH fe_tbl AS (

        SELECT
            DATE_FORMAT(DATE_PARSE(fee.yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            fee.city,
            fee.service_name,
            user_id as user_id,
            fee.fare_estimate_id,
            COALESCE(price_increase_strategy,'no_surge') AS strategy,
            sfe.session_service_level,
            sfe.session_id
        FROM
            hive.pricing.fare_estimates_enriched fee
        LEFT JOIN hive.marketplace.session_fare_estimates sfe    
            ON sfe.yyyymmdd = fee.yyyymmdd
            AND sfe.fare_estimate_id = fee.fare_estimate_id
            AND sfe.service_name = fee.service_name
        WHERE 
            fee.yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND fee.service_detail_id IN ('{service_detail_id_str}')
    ),

    rr_net_tbl as (

        SELECT   
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            city_name,
            service_obj_service_name service_name,
            service_detail_id,
            customer_id,
            customer_obj_mobile customer_mobile,
            estimate_id fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            lower(modified_order_status) modified_order_status
        FROM 
            orders.order_logs_fact
        WHERE
            yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND service_detail_id IN ('{service_detail_id_str}')
    ),

    fe_merged AS (
        SELECT
            fe_tbl.*,
            rr_net_tbl.customer_id,
            rr_net_tbl.order_id,
            rr_net_tbl.customer_mobile,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag,
            rr_net_tbl.modified_order_status
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.order_date = rr_net_tbl.order_date
            AND fe_tbl.user_id = rr_net_tbl.customer_id
            AND fe_tbl.fare_estimate_id = rr_net_tbl.fare_estimate_id
            AND fe_tbl.service_name = rr_net_tbl.service_name
    )

    SELECT 
        *
    FROM
    (
    SELECT
        user_id,

        MIN(customer_mobile) customer_mobile,

        COUNT(distinct fare_estimate_id) AS fe_count,
        COUNT(distinct order_id) AS rr_count,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

        COUNT(DISTINCT session_id) fe_session,
        COUNT(DISTINCT CASE WHEN order_id IS NOT NULL THEN session_id END) gross_sessions,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN session_id END) net_sessions,

        COUNT(DISTINCT CASE WHEN modified_order_status = 'cobrm' THEN order_id END) cobrm,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'cobra' THEN order_id END) cobra,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'ocara' THEN order_id END) ocara,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'expired' THEN order_id END) expired

    FROM 
        fe_merged
    GROUP BY 1
    )
    WHERE rr_count > 0
"""

df_customer_funnel = pd.read_sql(customer_funnel, connection)
df_customer_funnel.to_csv('customer_funnel_{}_to_{}.csv'\
                          .format(start_date, end_date),
                          index=False)

In [11]:
# df_customer_funnel.shape

##customer_segment

customer_segment = f"""

    WITH fe_tbl AS (
    
        SELECT
            customer_id user_id,
            city_name
        FROM
        (
        SELECT   
            customer_id,
            city_name,
            row_number() over(partition by customer_id order by epoch desc) seq_no
        FROM 
            orders.order_logs_fact
        WHERE
            yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND service_detail_id IN ('{service_detail_id_str}')
        )
        WHERE seq_no = 1
    )

          
            SELECT 
            
                    user_id ,
                    city_name,
                    gender_tag as  gender,
                    taxi_retention_segments ,
                    taxi_lifetime_stage,
                    taxi_ltr_segment,
                    taxi_service_affinity 
                 
            FROM 
                fe_tbl
            LEFT JOIN 
                datasets.iallocator_customer_segments
                ON user_id = customer_id 
                AND (run_date = '2024-06-25') 
            -- GROUP BY 1,2

"""

df_customer_segment = pd.read_sql(customer_segment, connection)

##Local copy
df_customer_segment.to_csv('customer_segment_{}_to_{}.csv' 
                           .format(start_date, end_date), index=False 
                          )

In [12]:
# df_customer_segment.shape

### Reading local extract

In [13]:
df_customer_segment = pd.read_csv('customer_segment_{}_to_{}.csv'.format(start_date, end_date))

In [14]:
df1 = pd.read_csv('customer_funnel_20240501_to_20240515.csv')
df2 = pd.read_csv('customer_funnel_20240516_to_20240531.csv')
df3 = pd.read_csv('customer_funnel_20240601_to_20240613.csv')
df4 = pd.read_csv('customer_funnel_20240614_to_20240626.csv')

df_customer_funnel_old = pd.concat([df1,df2,df3,df4])

In [15]:
print(df_customer_funnel_old.shape)
print(df_customer_funnel_old.user_id.nunique())
print(df_customer_segment.shape)
print(df_customer_segment.user_id.nunique())

(6974549, 12)
3481029
(3484926, 7)
3484926


In [16]:
df_customer_funnel_old.head(2)

,user_id,customer_mobile,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired
0,617bf46b3cbf28549db73b6b,9.945216e+09,11,6,4,5,4,4,0,0,0,2
1,61f61f2c6916b8f354aed5e5,8.073625e+09,8,5,2,5,4,2,0,2,1,0


In [17]:
df_customer_funnel_old['customer_mobile'].isnull().sum()

5

In [18]:
df_customer_funnel_cleand = df_customer_funnel_old.dropna(subset=['customer_mobile'])

In [19]:
df_customer_funnel_cleand['customer_mobile'] = df_customer_funnel_cleand['customer_mobile'].astype(float).astype(int)

In [20]:
df_customer_funnel = df_customer_funnel_cleand.groupby(['user_id', 'customer_mobile'])\
[['fe_count','rr_count','net_count','fe_session','gross_sessions','net_sessions','cobrm','cobra','ocara','expired',]]\
.agg('sum').reset_index()

In [21]:
df_exclude_user_id = df_customer_funnel\
.groupby('user_id')\
.agg(counts = ('customer_mobile', 'nunique'))\
.sort_values(by='counts', ascending=False)\
.reset_index()
df_exclude_user_id = df_exclude_user_id[df_exclude_user_id['counts'] == 2]

In [22]:
exclude_user_list = df_exclude_user_id['user_id'].tolist()
exclude_user_list

['6345857d76a9b3171fa5fd9e',
 '6481a6a5363c83f69be62880',
 '5d24e56716609737fa44767d',
 '6554048474f464906ca6e530',
 '64b92c49cdaae4585c2a4241',
 '64e846068b9ed49ec400c92f',
 '639aecddc2b501ef795c4c75',
 '5eb7654a5c16470e645917a2',
 '65858bdb59ffb827a35712c9',
 '5fe0ddc74319a729cfb3549c',
 '62f4e8824658002aabeef546',
 '624d2e9f3862620db120178d',
 '5ee216c49adf08f7cff7d51d',
 '57dfd36f8e934ecd1016857f',
 '5bdd506ea07a096e3e7ea854',
 '64eec09db937e56969797db8',
 '617626ff4c6ba14be4de467d',
 '5d2d9e6301197b4e4d195875',
 '61949833c98cb3f3f802286d',
 '653da9d78187d12b7bf31fc9',
 '65864e9ffe02b95e5c9cca63',
 '5d1683a0668011467e264c7e',
 '5d3eeebd9996733edb07547c',
 '635b88c9103aa964c4edfb88',
 '6441e5acce292832ab240b30',
 '5e391822fa5faf4ecf86f34f',
 '60868a01c14f0f8365ada9fa',
 '60432bbbaf535781d9242a26',
 '5a812f113a781a07c20abeb5',
 '5d20b78493686945e7161501',
 '5d0681590a565260304ab831',
 '5f9cb296ff48fe2033f451de',
 '5c7f6a388c352421eadbd21c',
 '61d51d59731a6d7abd05adca',
 '6544e47a3eff

In [23]:
df_customer_funnel_final = df_customer_funnel[~df_customer_funnel['user_id'].isin(exclude_user_list)]

In [24]:
df_customer_funnel_final.head(2)

,user_id,customer_mobile,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired
0,5737c6abddbec2203f733155,8971744217,3,2,1,3,2,1,0,1,0,0
1,5737c6adddbec2203f73316d,9916947141,15,4,2,6,4,2,0,0,1,0


In [25]:
df_customer_segment.head(2)

,user_id,city_name,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity
0,6259035f00b5f4cbe0ae5d8f,Bangalore,MALE,ELITE,HOOK,PHH,ONLY_LINK
1,5c066dc3b14ae33d3b9cb716,Bangalore,FEMALE,ELITE,COMMITTED,PHH,ONLY_LINK


In [26]:
## Joining DataFrames
df_customer_merge_raw = pd.merge(df_customer_funnel_final, df_customer_segment, how = 'left', on='user_id')
df_customer_merge = df_customer_merge_raw.copy()
df_customer_merge.head(5)

,user_id,customer_mobile,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,city_name,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity
0,5737c6abddbec2203f733155,8971744217,3,2,1,3,2,1,0,1,0,0,Bangalore,MALE,SILVER,HOOK,PHH,ONLY_AUTO
1,5737c6adddbec2203f73316d,9916947141,15,4,2,6,4,2,0,0,1,0,Bangalore,MALE,SILVER,HOOK,PHH,ONLY_LINK
2,5737c6aeddbec2203f733176,8073365587,58,26,14,28,19,14,0,4,2,6,Bangalore,MALE,PLATINUM,COMMITTED,PHH,ONLY_AUTO
3,5737c6b1ddbec2203f73318b,9035128750,4,2,2,4,2,2,0,0,0,0,Bangalore,MALE,SILVER,HOOK,PHH,LINK_AUTO
4,5737c6baddbec2203f7331d9,8553975420,4,3,2,3,3,2,0,0,1,0,Bangalore,MALE,SILVER,SUSTENANCE,PHH,LINK_AUTO


In [27]:
df_customer_merge['fe2rr'] = df_customer_merge['rr_count']*100.0/df_customer_merge['fe_count']
df_customer_merge['fe2net'] = df_customer_merge['net_count']*100.0/df_customer_merge['fe_count']
df_customer_merge['g2n'] = df_customer_merge['net_count']*100.0/df_customer_merge['rr_count']
df_customer_merge['g2n_session'] = df_customer_merge['net_sessions']*100.0/df_customer_merge['gross_sessions']
df_customer_merge['fe2rr_session'] = df_customer_merge['gross_sessions']*100.0/df_customer_merge['fe_session']
df_customer_merge['fe2net_session'] = df_customer_merge['net_sessions']*100.0/df_customer_merge['fe_session']
df_customer_merge = df_customer_merge.round(2)

In [28]:
df_customer_merge = df_customer_merge[['user_id', 'city_name', 'gender', 'customer_mobile',
                                       'fe_count', 'rr_count', 'net_count', 
                                       'fe2rr', 'g2n', 'fe2net',
                                       'fe_session','gross_sessions', 'net_sessions',
                                       'cobrm', 'cobra', 'ocara', 'expired',
                                       'fe2rr_session', 'g2n_session', 'fe2net_session',
                                       'taxi_retention_segments', 'taxi_lifetime_stage',
                                       'taxi_ltr_segment','taxi_service_affinity'
                                      ]]

In [29]:
df_customer_merge

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity
0,5737c6abddbec2203f733155,Bangalore,MALE,8971744217,3,2,1,66.67,50.00,33.33,3,2,1,0,1,0,0,66.67,50.00,33.33,SILVER,HOOK,PHH,ONLY_AUTO
1,5737c6adddbec2203f73316d,Bangalore,MALE,9916947141,15,4,2,26.67,50.00,13.33,6,4,2,0,0,1,0,66.67,50.00,33.33,SILVER,HOOK,PHH,ONLY_LINK
2,5737c6aeddbec2203f733176,Bangalore,MALE,8073365587,58,26,14,44.83,53.85,24.14,28,19,14,0,4,2,6,67.86,73.68,50.00,PLATINUM,COMMITTED,PHH,ONLY_AUTO
3,5737c6b1ddbec2203f73318b,Bangalore,MALE,9035128750,4,2,2,50.00,100.00,50.00,4,2,2,0,0,0,0,50.00,100.00,50.00,SILVER,HOOK,PHH,LINK_AUTO
4,5737c6baddbec2203f7331d9,Bangalore,MALE,8553975420,4,3,2,75.00,66.67,50.00,3,3,2,0,0,1,0,100.00,66.67,66.67,SILVER,SUSTENANCE,PHH,LINK_AUTO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3480965,667c5a8cd9b8f20accb751c8,Bangalore,NaN,8279435803,6,2,0,33.33,0.00,0.00,1,1,0,0,2,0,0,100.00,0.00,0.00,NaN,NaN,NaN,NaN
3480966,667c5ac1d9b8f29200b75980,Bangalore,NaN,9900002741,4,1,0,25.00,0.00,0.00,1,1,0,0,1,0,0,100.00,0.00,0.00,NaN,NaN,NaN,NaN
3480967,667c5b9e04165a2026c5b816,Bangalore,NaN,9957425825,4,1,0,25.00,0.00,0.00,1,1,0,0,0,1,0,100.00,0.00,0.00,NaN,NaN,NaN,NaN
3480968,667c5d078db9ae0efb10c0c8,Bangalore,NaN,9611204470,2,1,0,50.00,0.00,0.00,1,1,0,0,0,1,0,100.00,0.00,0.00,NaN,NaN,NaN,NaN


In [30]:
df_customer_merge[ df_customer_merge['user_id'].isin(['602b82723031acc8fb864739'])].T

,779350
user_id,602b82723031acc8fb864739
city_name,Bangalore
gender,MALE
customer_mobile,43718146
fe_count,6
rr_count,3
net_count,1
fe2rr,50.0
g2n,33.33
fe2net,16.67


#### Other EDA

In [31]:
df_customer_merge['length'] = df_customer_merge['customer_mobile'].astype(str).apply(len)

In [32]:
df_customer_merge.head(10)

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,length
0,5737c6abddbec2203f733155,Bangalore,MALE,8971744217,3,2,1,66.67,50.00,33.33,3,2,1,0,1,0,0,66.67,50.00,33.33,SILVER,HOOK,PHH,ONLY_AUTO,10
1,5737c6adddbec2203f73316d,Bangalore,MALE,9916947141,15,4,2,26.67,50.00,13.33,6,4,2,0,0,1,0,66.67,50.00,33.33,SILVER,HOOK,PHH,ONLY_LINK,10
2,5737c6aeddbec2203f733176,Bangalore,MALE,8073365587,58,26,14,44.83,53.85,24.14,28,19,14,0,4,2,6,67.86,73.68,50.00,PLATINUM,COMMITTED,PHH,ONLY_AUTO,10
3,5737c6b1ddbec2203f73318b,Bangalore,MALE,9035128750,4,2,2,50.00,100.00,50.00,4,2,2,0,0,0,0,50.00,100.00,50.00,SILVER,HOOK,PHH,LINK_AUTO,10
4,5737c6baddbec2203f7331d9,Bangalore,MALE,8553975420,4,3,2,75.00,66.67,50.00,3,3,2,0,0,1,0,100.00,66.67,66.67,SILVER,SUSTENANCE,PHH,LINK_AUTO,10
5,5737c6bbddbec2203f7331e5,Bangalore,MALE,9901343947,1,1,1,100.00,100.00,100.00,1,1,1,0,0,0,0,100.00,100.00,100.00,HH,HANDHOLDING,HH,ONLY_LINK,10
6,5737c6bbddbec2203f7331e8,Bangalore,MALE,9916539554,18,4,3,22.22,75.00,16.67,7,4,3,0,0,0,1,57.14,75.00,42.86,PLATINUM,COMMITTED,PHH,ONLY_LINK,10
7,5737c6c0ddbec2203f733215,Bangalore,MALE,8553478009,10,4,2,40.00,50.00,20.00,3,3,2,0,0,0,2,100.00,66.67,66.67,DORMANT,DORMANT,PHH,ONLY_LINK,10
8,5737c6c2ddbec2203f733227,Bangalore,MALE,9620465588,4,1,1,25.00,100.00,25.00,2,1,1,0,0,0,0,50.00,100.00,50.00,HH,HANDHOLDING,HH,ONLY_CAB,10
9,5737c6c5ddbec2203f733239,Bangalore,MALE,8105183288,1,1,1,100.00,100.00,100.00,1,1,1,0,0,0,0,100.00,100.00,100.00,DORMANT,DORMANT,PHH,UNKNOWN,10


In [33]:
# df_stratified_random_sampling = df_customer_merge \
#                                     [df_customer_merge['rr_count'] > 0]
# df_stratified_random_sampling.head(5)

In [34]:
df_stratified_random_sampling = df_customer_merge[df_customer_merge['length'] == 10].copy()

In [35]:
print(df_stratified_random_sampling.shape)
print(df_stratified_random_sampling.user_id.nunique())

(3480943, 25)
3480943


In [36]:
df_stratified_random_sampling.city_name.unique()

array(['Bangalore', nan], dtype=object)

In [37]:
df_stratified_random_sampling.groupby(['city_name']).user_id.nunique()

city_name
Bangalore    3480721
Name: user_id, dtype: int64

In [38]:
df_stratified_random_sampling = df_stratified_random_sampling[df_stratified_random_sampling['city_name']\
                                                              .isin(['Bangalore'])]

In [39]:
df_stratified_random_sampling['customer_mobile'] = df_stratified_random_sampling['customer_mobile'].apply(str)

In [40]:
df_stratified_random_sampling.head(5)

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,length
0,5737c6abddbec2203f733155,Bangalore,MALE,8971744217,3,2,1,66.67,50.00,33.33,3,2,1,0,1,0,0,66.67,50.00,33.33,SILVER,HOOK,PHH,ONLY_AUTO,10
1,5737c6adddbec2203f73316d,Bangalore,MALE,9916947141,15,4,2,26.67,50.00,13.33,6,4,2,0,0,1,0,66.67,50.00,33.33,SILVER,HOOK,PHH,ONLY_LINK,10
2,5737c6aeddbec2203f733176,Bangalore,MALE,8073365587,58,26,14,44.83,53.85,24.14,28,19,14,0,4,2,6,67.86,73.68,50.00,PLATINUM,COMMITTED,PHH,ONLY_AUTO,10
3,5737c6b1ddbec2203f73318b,Bangalore,MALE,9035128750,4,2,2,50.00,100.00,50.00,4,2,2,0,0,0,0,50.00,100.00,50.00,SILVER,HOOK,PHH,LINK_AUTO,10
4,5737c6baddbec2203f7331d9,Bangalore,MALE,8553975420,4,3,2,75.00,66.67,50.00,3,3,2,0,0,1,0,100.00,66.67,66.67,SILVER,SUSTENANCE,PHH,LINK_AUTO,10


In [41]:
df_stratified_random_sampling[['city_name', 'gender',
                               'fe_session','gross_sessions', 'net_sessions',
                               'cobrm', 'cobra', 'ocara', 'expired',
                               'fe2rr_session', 'g2n_session', 'fe2net_session',
                               'taxi_retention_segments' ,'taxi_lifetime_stage','taxi_ltr_segment','taxi_service_affinity'
                              ]].isnull().sum()

city_name                     0
gender                     7502
fe_session                    0
gross_sessions                0
net_sessions                  0
cobrm                         0
cobra                         0
ocara                         0
expired                       0
fe2rr_session                 0
g2n_session                2917
fe2net_session                0
taxi_retention_segments    7352
taxi_lifetime_stage        7352
taxi_ltr_segment           7352
taxi_service_affinity      7352
dtype: int64

In [42]:
# # Columns to fill null values in
# columns_to_fill = ['lifetime_stage_jan1', 'lifetime_stage_jan21',
#                    'recency_segment_jan1', 'recency_segment_jan21', 
#                    'service_affinity_jan1', 'service_affinity_jan21']

# # Fill null values in the specified columns with a specific value (e.g., 0)
# df_stratified_random_sampling[columns_to_fill] = df_stratified_random_sampling[columns_to_fill].fillna('NA')

In [43]:
df_stratified_random_sampling[['city_name', 'gender',
                               'fe_session','gross_sessions', 'net_sessions',
                               'cobrm', 'cobra', 'ocara', 'expired',
                               'fe2rr_session', 'g2n_session', 'fe2net_session',
                               'taxi_retention_segments' ,'taxi_lifetime_stage','taxi_ltr_segment','taxi_service_affinity'
                              ]].isnull().sum()

city_name                     0
gender                     7502
fe_session                    0
gross_sessions                0
net_sessions                  0
cobrm                         0
cobra                         0
ocara                         0
expired                       0
fe2rr_session                 0
g2n_session                2917
fe2net_session                0
taxi_retention_segments    7352
taxi_lifetime_stage        7352
taxi_ltr_segment           7352
taxi_service_affinity      7352
dtype: int64

In [44]:
# df_stratified_random_sampling = df_stratified_random_sampling[df_stratified_random_sampling['g2n_session'].notna()]

In [45]:
df_stratified_random_sampling[['city_name', 'gender',
                               'fe_session','gross_sessions', 'net_sessions',
                               'cobrm', 'cobra', 'ocara', 'expired',
                               'fe2rr_session', 'g2n_session', 'fe2net_session',
                               'taxi_retention_segments' ,'taxi_lifetime_stage','taxi_ltr_segment','taxi_service_affinity'
                              ]].isnull().sum()

city_name                     0
gender                     7502
fe_session                    0
gross_sessions                0
net_sessions                  0
cobrm                         0
cobra                         0
ocara                         0
expired                       0
fe2rr_session                 0
g2n_session                2917
fe2net_session                0
taxi_retention_segments    7352
taxi_lifetime_stage        7352
taxi_ltr_segment           7352
taxi_service_affinity      7352
dtype: int64

In [46]:
df_stratified_random_sampling.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3480721 entries, 0 to 3480969
Data columns (total 25 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   user_id                  object 
 1   city_name                object 
 2   gender                   object 
 3   customer_mobile          object 
 4   fe_count                 int64  
 5   rr_count                 int64  
 6   net_count                int64  
 7   fe2rr                    float64
 8   g2n                      float64
 9   fe2net                   float64
 10  fe_session               int64  
 11  gross_sessions           int64  
 12  net_sessions             int64  
 13  cobrm                    int64  
 14  cobra                    int64  
 15  ocara                    int64  
 16  expired                  int64  
 17  fe2rr_session            float64
 18  g2n_session              float64
 19  fe2net_session           float64
 20  taxi_retention_segments  object 
 21  taxi_lifetime

In [47]:
df_customer_merge[ df_customer_merge['user_id'].isin(['631216f8f9a036dc3c412ac4'])].T

,1659187
user_id,631216f8f9a036dc3c412ac4
city_name,Bangalore
gender,FEMALE
customer_mobile,9334465973
fe_count,90
rr_count,42
net_count,22
fe2rr,46.67
g2n,52.38
fe2net,24.44


## Stratified Random Sampling

In [48]:
df_stratified_random_sampling.shape

(3480721, 25)

In [49]:
df_stratified_random_sampling.user_id.nunique()

3480721

In [50]:
df_stratified_random_sampling['customer_mobile'] = pd.to_numeric(df_stratified_random_sampling['customer_mobile'], errors='coerce').astype(float)

# Drop rows with NaN values in any column
df_stratified_random_sampling_cleaned = df_stratified_random_sampling.dropna()

In [51]:
df_stratified_random_sampling_cleaned.shape

(3470305, 25)

In [52]:
df_stratified_random_sampling_cleaned.user_id.nunique()

3470305

In [53]:
stratify_grouper_col =['fe_session','gross_sessions', 'net_sessions',
                       'cobrm', 'cobra', 'ocara', 'expired',
                      ]

In [54]:
random_seed = 42

# Split the df into two parts for initial 'control' and 'temp'
df_train_control, df_test = train_test_split(df_stratified_random_sampling_cleaned, 
                                             test_size=1/2, 
                                             random_state=random_seed
#                                              stratify=df_stratified_random_sampling_cleaned[stratify_grouper_col]
                                            )



In [55]:

df_train_control = df_train_control.sample(min(400000, len(df_train_control)), random_state=random_seed)
df_test = df_test.sample(min(400000, len(df_test)), random_state=random_seed)
#df_test2 = df_test2.sample(min(300000, len(df_test2)), random_state=random_seed)

# Assign group labels
df_train_control['group_tc'] = 'control'
df_test['group_tc'] = 'test'

df_final_sample = pd.concat([df_train_control, df_test])

In [56]:
print(df_stratified_random_sampling_cleaned.shape)
print(df_final_sample.shape)

(3470305, 25)
(800000, 26)


In [57]:
df_final_sample

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,length,group_tc
2688545,656ef08eec39c6303bbca634,Bangalore,MALE,9.740415e+09,15,7,4,46.67,57.14,26.67,11,7,4,0,2,1,0,63.64,57.14,36.36,DORMANT,DORMANT,PHH,ONLY_AUTO,10,control
1477344,62c3e86017af9ad5a66a8dcf,Bangalore,MALE,9.347912e+09,8,3,0,37.50,0.00,0.00,1,1,0,0,3,0,0,100.00,0.00,0.00,INACTIVE,INACTIVE,PHH,ONLY_AUTO,10,control
2195283,644d2eb30ffc1680ac19c46e,Bangalore,FEMALE,8.073490e+09,43,25,14,58.14,56.00,32.56,24,19,14,0,5,2,4,79.17,73.68,58.33,PLATINUM,COMMITTED,PHH,ONLY_LINK,10,control
412502,5d57d31455fbf50d45f9dce1,Bangalore,MALE,7.829076e+09,25,4,1,16.00,25.00,4.00,4,4,1,0,2,1,0,100.00,25.00,25.00,DORMANT,DORMANT,PHH,ONLY_LINK,10,control
1923961,639dd3b0cd56c026eb837257,Bangalore,MALE,9.936613e+09,2,2,1,100.00,50.00,50.00,2,2,1,0,0,0,1,100.00,50.00,50.00,SILVER,HOOK,PHH,ONLY_AUTO,10,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2608260,653fd038a385937397c17b44,Bangalore,FEMALE,9.566324e+09,2,2,2,100.00,100.00,100.00,2,2,2,0,0,0,0,100.00,100.00,100.00,SILVER,HOOK,PHH,ONLY_AUTO,10,test
949954,618646ecb281d714aac7155a,Bangalore,FEMALE,8.714805e+09,107,48,13,44.86,27.08,12.15,43,38,13,0,3,18,14,88.37,34.21,30.23,PLATINUM,COMMITTED,PHH,NO_AFFINITY,10,test
1221006,62576c2591957fd136927904,Bangalore,FEMALE,8.317438e+09,17,7,5,41.18,71.43,29.41,9,7,5,0,0,2,0,77.78,71.43,55.56,GOLD,CHURN_OTB,PHH,ONLY_AUTO,10,test
3060915,661c7d1d2a30115c48a27fa8,Bangalore,MALE,9.148075e+09,1,1,1,100.00,100.00,100.00,1,1,1,0,0,0,0,100.00,100.00,100.00,DORMANT,DORMANT,HH,ONLY_AUTO,10,test


In [58]:
df_final_sample['customer_mobile'].nunique()

800000

### Balance Groups

In [59]:
df_final_sample.groupby(['group_tc'])\
.agg(user_base = pd.NamedAgg('user_id', 'nunique'),
     customer_mobile = pd.NamedAgg('customer_mobile', 'nunique'),
     city_distr = pd.NamedAgg(column='city_name', 
                              aggfunc=lambda x: x.value_counts().to_dict()),
     fe_count = pd.NamedAgg('fe_count', 'sum'),
     rr_count = pd.NamedAgg('rr_count', 'sum'),
     net_count = pd.NamedAgg('net_count', 'sum'),
     fe_session = pd.NamedAgg('fe_session', 'sum'),
     gross_sessions = pd.NamedAgg('gross_sessions', 'sum'),
     net_sessions = pd.NamedAgg('net_sessions', 'sum'),
     cobrm = pd.NamedAgg('cobrm', 'sum'),
     cobra = pd.NamedAgg('cobra', 'sum'),
     ocara = pd.NamedAgg('ocara', 'sum'),
     expired = pd.NamedAgg('expired', 'sum'),

     retention_segment_dist = pd.NamedAgg(column='taxi_retention_segments', 
                                        aggfunc=lambda x: x.value_counts().to_dict()), 
     
     lifetime_stage_dist = pd.NamedAgg(column='taxi_lifetime_stage', 
                                        aggfunc=lambda x: x.value_counts().to_dict()), 
     
     ltr_segment_dist = pd.NamedAgg(column='taxi_ltr_segment', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     service_affinity_dist = pd.NamedAgg(column='taxi_service_affinity', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),
     
     gender_distr = pd.NamedAgg(column='gender', 
                                aggfunc=lambda x: x.value_counts().to_dict()),
        
    ).T

group_tc,control,test
user_base,400000,400000
customer_mobile,400000,400000
city_distr,{'Bangalore': 400000},{'Bangalore': 400000}
fe_count,9795995,9785170
rr_count,4039480,4036436
net_count,1799317,1797741
fe_session,4318398,4325640
gross_sessions,2969434,2968630
net_sessions,1798207,1796597
cobrm,24085,24036


In [60]:
df_final_sample_city = df_final_sample.groupby(['city_name', 'group_tc'])\
.agg(user_base = pd.NamedAgg('user_id', 'nunique'),
     fe_count = pd.NamedAgg('fe_count', 'sum'),
     rr_count = pd.NamedAgg('rr_count', 'sum'),
     net_count = pd.NamedAgg('net_count', 'sum'),
     fe_session = pd.NamedAgg('fe_session', 'sum'),
     gross_sessions = pd.NamedAgg('gross_sessions', 'sum'),
     net_sessions = pd.NamedAgg('net_sessions', 'sum'),
     cobrm = pd.NamedAgg('cobrm', 'sum'),
     cobra = pd.NamedAgg('cobra', 'sum'),
     ocara = pd.NamedAgg('ocara', 'sum'),
     expired = pd.NamedAgg('expired', 'sum'),

     retention_segment_dist = pd.NamedAgg(column='taxi_retention_segments', 
                                        aggfunc=lambda x: x.value_counts().to_dict()), 
     
     lifetime_stage_dist = pd.NamedAgg(column='taxi_lifetime_stage', 
                                        aggfunc=lambda x: x.value_counts().to_dict()), 
     
     ltr_segment_dist = pd.NamedAgg(column='taxi_ltr_segment', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     service_affinity_dist = pd.NamedAgg(column='taxi_service_affinity', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     gender_distr = pd.NamedAgg(column='gender', 
                                aggfunc=lambda x: x.value_counts().to_dict()),
     
    ).reset_index()
df_final_sample_city

,city_name,group_tc,user_base,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,retention_segment_dist,lifetime_stage_dist,ltr_segment_dist,service_affinity_dist,gender_distr
0,Bangalore,control,400000,9795995,4039480,1799317,4318398,2969434,1798207,24085,1029196,587681,588651,"{'DORMANT': 137386, 'GOLD': 61382, 'SILVER': 51223, 'PLATINUM': 45889, 'HH': 36360, 'INACTIVE': 26711, 'ELITE': 22512, 'UNKNOWN': 17290, 'PRIME': 1247}","{'DORMANT': 137386, 'HOOK': 69065, 'COMMITTED': 50098, 'HANDHOLDING': 36360, 'SUSTENANCE': 35465, 'INACTIVE': 26711, 'UNKNOWN': 17290, 'CHURN_OTB': 16243, 'SOFT_CHURN': 11047, 'DETOX': 335}","{'PHH': 293075, 'HH': 89635, 'UNKNOWN': 17290}","{'ONLY_AUTO': 166792, 'ONLY_LINK': 98082, 'LINK_AUTO': 36799, 'NO_AFFINITY': 34252, 'UNKNOWN': 28876, 'ONLY_CAB': 18761, 'AUTO_CAB': 13451, 'LINK_CAB': 2987}","{'MALE': 277649, 'FEMALE': 111704, 'UNKNOWN': 9400, 'OTHERS': 1247}"
1,Bangalore,test,400000,9785170,4036436,1797741,4325640,2968630,1796597,24036,1026014,584588,593543,"{'DORMANT': 137821, 'GOLD': 61132, 'SILVER': 51472, 'PLATINUM': 45691, 'HH': 36450, 'INACTIVE': 26536, 'ELITE': 22421, 'UNKNOWN': 17198, 'PRIME': 1279}","{'DORMANT': 137821, 'HOOK': 69428, 'COMMITTED': 50113, 'HANDHOLDING': 36450, 'SUSTENANCE': 35161, 'INACTIVE': 26536, 'UNKNOWN': 17198, 'CHURN_OTB': 15851, 'SOFT_CHURN': 11077, 'DETOX': 365}","{'PHH': 293102, 'HH': 89700, 'UNKNOWN': 17198}","{'ONLY_AUTO': 166981, 'ONLY_LINK': 98175, 'LINK_AUTO': 36478, 'NO_AFFINITY': 34269, 'UNKNOWN': 28849, 'ONLY_CAB': 18749, 'AUTO_CAB': 13488, 'LINK_CAB': 3011}","{'MALE': 277507, 'FEMALE': 111727, 'UNKNOWN': 9472, 'OTHERS': 1294}"


In [61]:
df_final_sample_city.columns

Index(['city_name', 'group_tc', 'user_base', 'fe_count', 'rr_count',
       'net_count', 'fe_session', 'gross_sessions', 'net_sessions', 'cobrm',
       'cobra', 'ocara', 'expired', 'retention_segment_dist',
       'lifetime_stage_dist', 'ltr_segment_dist', 'service_affinity_dist',
       'gender_distr'],
      dtype='object')

In [62]:
df_final_sample_city['fe2rr'] = df_final_sample_city['rr_count']*100.0/df_final_sample_city['fe_count']
df_final_sample_city['fe2net'] = df_final_sample_city['net_count']*100.0/df_final_sample_city['fe_count']
df_final_sample_city['g2n'] = df_final_sample_city['net_count']*100.0/df_final_sample_city['rr_count']
df_final_sample_city['g2n_session'] = df_final_sample_city['net_sessions']*100.0/df_final_sample_city['gross_sessions']
df_final_sample_city['fe2rr_session'] = df_final_sample_city['gross_sessions']*100.0/df_final_sample_city['fe_session']
df_final_sample_city['fe2net_session'] = df_final_sample_city['net_sessions']*100.0/df_final_sample_city['fe_session']

In [63]:
df_final_sample_city

,city_name,group_tc,user_base,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,retention_segment_dist,lifetime_stage_dist,ltr_segment_dist,service_affinity_dist,gender_distr,fe2rr,fe2net,g2n,g2n_session,fe2rr_session,fe2net_session
0,Bangalore,control,400000,9795995,4039480,1799317,4318398,2969434,1798207,24085,1029196,587681,588651,"{'DORMANT': 137386, 'GOLD': 61382, 'SILVER': 51223, 'PLATINUM': 45889, 'HH': 36360, 'INACTIVE': 26711, 'ELITE': 22512, 'UNKNOWN': 17290, 'PRIME': 1247}","{'DORMANT': 137386, 'HOOK': 69065, 'COMMITTED': 50098, 'HANDHOLDING': 36360, 'SUSTENANCE': 35465, 'INACTIVE': 26711, 'UNKNOWN': 17290, 'CHURN_OTB': 16243, 'SOFT_CHURN': 11047, 'DETOX': 335}","{'PHH': 293075, 'HH': 89635, 'UNKNOWN': 17290}","{'ONLY_AUTO': 166792, 'ONLY_LINK': 98082, 'LINK_AUTO': 36799, 'NO_AFFINITY': 34252, 'UNKNOWN': 28876, 'ONLY_CAB': 18761, 'AUTO_CAB': 13451, 'LINK_CAB': 2987}","{'MALE': 277649, 'FEMALE': 111704, 'UNKNOWN': 9400, 'OTHERS': 1247}",41.236036,18.367884,44.543283,60.557231,68.762398,41.640604
1,Bangalore,test,400000,9785170,4036436,1797741,4325640,2968630,1796597,24036,1026014,584588,593543,"{'DORMANT': 137821, 'GOLD': 61132, 'SILVER': 51472, 'PLATINUM': 45691, 'HH': 36450, 'INACTIVE': 26536, 'ELITE': 22421, 'UNKNOWN': 17198, 'PRIME': 1279}","{'DORMANT': 137821, 'HOOK': 69428, 'COMMITTED': 50113, 'HANDHOLDING': 36450, 'SUSTENANCE': 35161, 'INACTIVE': 26536, 'UNKNOWN': 17198, 'CHURN_OTB': 15851, 'SOFT_CHURN': 11077, 'DETOX': 365}","{'PHH': 293102, 'HH': 89700, 'UNKNOWN': 17198}","{'ONLY_AUTO': 166981, 'ONLY_LINK': 98175, 'LINK_AUTO': 36478, 'NO_AFFINITY': 34269, 'UNKNOWN': 28849, 'ONLY_CAB': 18749, 'AUTO_CAB': 13488, 'LINK_CAB': 3011}","{'MALE': 277507, 'FEMALE': 111727, 'UNKNOWN': 9472, 'OTHERS': 1294}",41.250545,18.372098,44.537830,60.519398,68.628688,41.533669


In [64]:
df_final_sample.reset_index(drop=True)

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,length,group_tc
0,656ef08eec39c6303bbca634,Bangalore,MALE,9.740415e+09,15,7,4,46.67,57.14,26.67,11,7,4,0,2,1,0,63.64,57.14,36.36,DORMANT,DORMANT,PHH,ONLY_AUTO,10,control
1,62c3e86017af9ad5a66a8dcf,Bangalore,MALE,9.347912e+09,8,3,0,37.50,0.00,0.00,1,1,0,0,3,0,0,100.00,0.00,0.00,INACTIVE,INACTIVE,PHH,ONLY_AUTO,10,control
2,644d2eb30ffc1680ac19c46e,Bangalore,FEMALE,8.073490e+09,43,25,14,58.14,56.00,32.56,24,19,14,0,5,2,4,79.17,73.68,58.33,PLATINUM,COMMITTED,PHH,ONLY_LINK,10,control
3,5d57d31455fbf50d45f9dce1,Bangalore,MALE,7.829076e+09,25,4,1,16.00,25.00,4.00,4,4,1,0,2,1,0,100.00,25.00,25.00,DORMANT,DORMANT,PHH,ONLY_LINK,10,control
4,639dd3b0cd56c026eb837257,Bangalore,MALE,9.936613e+09,2,2,1,100.00,50.00,50.00,2,2,1,0,0,0,1,100.00,50.00,50.00,SILVER,HOOK,PHH,ONLY_AUTO,10,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799995,653fd038a385937397c17b44,Bangalore,FEMALE,9.566324e+09,2,2,2,100.00,100.00,100.00,2,2,2,0,0,0,0,100.00,100.00,100.00,SILVER,HOOK,PHH,ONLY_AUTO,10,test
799996,618646ecb281d714aac7155a,Bangalore,FEMALE,8.714805e+09,107,48,13,44.86,27.08,12.15,43,38,13,0,3,18,14,88.37,34.21,30.23,PLATINUM,COMMITTED,PHH,NO_AFFINITY,10,test
799997,62576c2591957fd136927904,Bangalore,FEMALE,8.317438e+09,17,7,5,41.18,71.43,29.41,9,7,5,0,0,2,0,77.78,71.43,55.56,GOLD,CHURN_OTB,PHH,ONLY_AUTO,10,test
799998,661c7d1d2a30115c48a27fa8,Bangalore,MALE,9.148075e+09,1,1,1,100.00,100.00,100.00,1,1,1,0,0,0,0,100.00,100.00,100.00,DORMANT,DORMANT,HH,ONLY_AUTO,10,test


In [65]:
##Local extract

df_final_sample.to_csv( 'scratch_card_direct_open_exp_group_tc.csv', index=False )

In [66]:
# Check for common values in 'user_id' among the three DataFrames
common_values_df1_df2 = df_train_control[df_train_control['user_id'].isin(df_test['user_id'])]['user_id'].tolist()
common_values_df1_df2

[]

In [67]:
print(df_train_control.shape)
print(df_test.shape)

print(df_train_control.user_id.nunique())
print(df_test.user_id.nunique())


(400000, 26)
(400000, 26)
400000
400000


In [68]:
df_train_control[df_train_control['user_id'].isin(df_test['user_id'])]

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,length,group_tc


In [69]:
df_test[df_test['user_id'].isin(df_train_control['user_id'])]

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,length,group_tc


## User Selector for Experiment

In [73]:
df_train_control.reset_index(drop=True)

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,length,group_tc
0,656ef08eec39c6303bbca634,Bangalore,MALE,9740415139,15,7,4,46.67,57.14,26.67,11,7,4,0,2,1,0,63.64,57.14,36.36,DORMANT,DORMANT,PHH,ONLY_AUTO,10,control
1,62c3e86017af9ad5a66a8dcf,Bangalore,MALE,9347911593,8,3,0,37.50,0.00,0.00,1,1,0,0,3,0,0,100.00,0.00,0.00,INACTIVE,INACTIVE,PHH,ONLY_AUTO,10,control
2,644d2eb30ffc1680ac19c46e,Bangalore,FEMALE,8073489621,43,25,14,58.14,56.00,32.56,24,19,14,0,5,2,4,79.17,73.68,58.33,PLATINUM,COMMITTED,PHH,ONLY_LINK,10,control
3,5d57d31455fbf50d45f9dce1,Bangalore,MALE,7829075533,25,4,1,16.00,25.00,4.00,4,4,1,0,2,1,0,100.00,25.00,25.00,DORMANT,DORMANT,PHH,ONLY_LINK,10,control
4,639dd3b0cd56c026eb837257,Bangalore,MALE,9936612966,2,2,1,100.00,50.00,50.00,2,2,1,0,0,0,1,100.00,50.00,50.00,SILVER,HOOK,PHH,ONLY_AUTO,10,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,632ea536c019fc31ec3d4633,Bangalore,MALE,8897492518,80,34,10,42.50,29.41,12.50,31,26,10,0,13,9,2,83.87,38.46,32.26,PLATINUM,COMMITTED,PHH,ONLY_AUTO,10,control
399996,64fe070a89d1cc78d09cd27d,Bangalore,MALE,9353032714,1,1,1,100.00,100.00,100.00,1,1,1,0,0,0,0,100.00,100.00,100.00,SILVER,HOOK,PHH,ONLY_AUTO,10,control
399997,666a9adf620d068605257797,Bangalore,MALE,9019126574,9,3,2,33.33,66.67,22.22,3,2,2,0,1,0,0,66.67,100.00,66.67,HH,HANDHOLDING,HH,LINK_AUTO,10,control
399998,665e871f96771d61ef88142d,Bangalore,FEMALE,9916896612,6,2,2,33.33,100.00,33.33,5,2,2,0,0,0,0,40.00,100.00,40.00,DORMANT,DORMANT,HH,ONLY_AUTO,10,control


In [75]:
df_test.reset_index(drop=True)

,user_id,city_name,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,fe2rr_session,g2n_session,fe2net_session,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,length,group_tc
0,630619653fb5f523c4e7f61b,Bangalore,UNKNOWN,9818461884,4,1,1,25.00,100.00,25.00,2,1,1,0,0,0,0,50.00,100.00,50.00,DORMANT,DORMANT,PHH,ONLY_LINK,10,test
1,623c8acbb9a21e0575cd069e,Bangalore,MALE,8553542956,9,4,1,44.44,25.00,11.11,7,4,1,0,1,1,1,57.14,25.00,14.29,SILVER,HOOK,PHH,ONLY_AUTO,10,test
2,6321837f6837d2a173120179,Bangalore,MALE,8057445838,5,2,0,40.00,0.00,0.00,1,1,0,0,0,2,0,100.00,0.00,0.00,DORMANT,DORMANT,PHH,ONLY_AUTO,10,test
3,643bda628786685836627f3d,Bangalore,FEMALE,6305601696,2,1,0,50.00,0.00,0.00,2,1,0,0,0,1,0,50.00,0.00,0.00,INACTIVE,INACTIVE,PHH,ONLY_AUTO,10,test
4,664f4a4922cc5d7aa80308a5,Bangalore,FEMALE,7259307404,6,2,0,33.33,0.00,0.00,2,2,0,0,0,1,1,100.00,0.00,0.00,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,10,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,653fd038a385937397c17b44,Bangalore,FEMALE,9566324146,2,2,2,100.00,100.00,100.00,2,2,2,0,0,0,0,100.00,100.00,100.00,SILVER,HOOK,PHH,ONLY_AUTO,10,test
399996,618646ecb281d714aac7155a,Bangalore,FEMALE,8714804780,107,48,13,44.86,27.08,12.15,43,38,13,0,3,18,14,88.37,34.21,30.23,PLATINUM,COMMITTED,PHH,NO_AFFINITY,10,test
399997,62576c2591957fd136927904,Bangalore,FEMALE,8317437651,17,7,5,41.18,71.43,29.41,9,7,5,0,0,2,0,77.78,71.43,55.56,GOLD,CHURN_OTB,PHH,ONLY_AUTO,10,test
399998,661c7d1d2a30115c48a27fa8,Bangalore,MALE,9148075494,1,1,1,100.00,100.00,100.00,1,1,1,0,0,0,0,100.00,100.00,100.00,DORMANT,DORMANT,HH,ONLY_AUTO,10,test


In [72]:
df_train_control['customer_mobile'] = df_train_control['customer_mobile'].astype(float).astype(int)

In [74]:
df_test['customer_mobile'] = df_test['customer_mobile'].astype(float).astype(int)

In [77]:
def split_fn(df, type):

    type = type
    total_rows = len(df)
    chunk_size = total_rows // 4
    
    for i in range(4):
        start_row = i * chunk_size
        end_row = (i + 1) * chunk_size
        df_chunk = df.iloc[start_row:end_row]
        df_chunk.to_csv(f'deals_experiment_{type}_{i+1}.csv'.format(), 
                        header=False,
                        index=False)

    return print('done')

In [78]:
split_fn(df_train_control[['customer_mobile']], 'control')

done


In [79]:
split_fn(df_test[['customer_mobile']], 'test')

done


In [ ]:
# df_train_control['customer_mobile'] = df_train_control['customer_mobile'].astype(float).astype(int)
# df_train_control[['customer_mobile']].to_csv('deals_experiment_control.csv', 
#                                              header=False,
#                                              index=False )

In [ ]:
# df_test['customer_mobile'] = df_test['customer_mobile'].astype(float).astype(int)
# df_test[['customer_mobile']].to_csv('deals_experiment_test.csv', 
#                                      header=False,
#                                      index=False )